# ALL Detection — Google Colab Runner

**Çalıştırma sırası:** hücreleri yukarıdan aşağıya sırayla çalıştırın.

| Hücre | Faz | İşlem |
|-------|-----|-------|
| 1 | — | GPU & donanım kontrolü |
| 2 | — | Google Drive bağlama |
| 3 | — | Repo klonlama & pip kurulum |
| 4 | 1 | Kaggle kimlik doğrulama & veri indirme |
| 5 | 1 | Veri keşfi (EDA) |
| 6 | 1 | Hasta-bazlı train/val/test bölme |
| 7 | 3/4 | Model eğitimi |
| 8 | 5 | Değerlendirme & karşılaştırma |

## Hücre 1 — GPU & Donanım Kontrolü

In [ ]:
import torch, platform, json, os
from pathlib import Path

info = {
    "platform":       platform.platform(),
    "python":         platform.python_version(),
    "torch":          torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu_name":       torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A",
    "gpu_memory_gb":  round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
                      if torch.cuda.is_available() else 0,
    "cuda_version":   torch.version.cuda or "N/A",
}

for k, v in info.items():
    print(f"{k:20s}: {v}")

assert torch.cuda.is_available(), "GPU bulunamadı! Runtime > Change runtime type > GPU seçin."

## Hücre 2 — Google Drive Bağlama (Önerilen)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Repo Drive'da saklanacaksa bu yolu kullanın:
# REPO_ROOT = '/content/drive/MyDrive/all-detection-final'
# Sadece Colab geçici depolama (oturum kapanırsa kaybolur):
REPO_ROOT = '/content/all-detection-final'

print(f'Repo kökü: {REPO_ROOT}')

## Hücre 3 — Repo Klonlama & Bağımlılık Kurulumu

In [ ]:
import os, subprocess

GITHUB_URL = 'https://github.com/ysftrv/all-detection-final.git'

if not os.path.exists(REPO_ROOT):
    subprocess.run(['git', 'clone', GITHUB_URL, REPO_ROOT], check=True)
else:
    subprocess.run(['git', '-C', REPO_ROOT, 'pull'], check=True)

os.chdir(REPO_ROOT)
print(f'Çalışma dizini: {os.getcwd()}')

subprocess.run(['pip', 'install', '-r', 'requirements.txt', '-q'], check=True)
print('Kurulum tamamlandı.')

## Hücre 4 — Kaggle Kimlik Doğrulama & Veri İndirme (Faz 1)

In [ ]:
# --- Kimlik doğrulama ---
# Yöntem A (Colab Secret — önerilen, güvenli):
#   Colab sol paneli → 🔑 Secrets → KAGGLE_USERNAME ve KAGGLE_KEY ekleyin.
from google.colab import userdata
import os, json
from pathlib import Path

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
cred_path = kaggle_dir / 'kaggle.json'
cred_path.write_text(json.dumps({
    'username': userdata.get('KAGGLE_USERNAME'),
    'key':      userdata.get('KAGGLE_KEY'),
}))
cred_path.chmod(0o600)
print('Kaggle kimlik bilgisi ayarlandı.')

# Yöntem B (manuel yükleme — Secrets yoksa):
# from google.colab import files
# files.upload()  # kaggle.json dosyasını seçin
# import shutil; shutil.move('kaggle.json', str(cred_path)); cred_path.chmod(0o600)

In [ ]:
# --- Veri İndirme ---
# İdempotent: veri zaten varsa tekrar indirmez.
import sys
sys.path.insert(0, REPO_ROOT)

from src.data.download import download
dataset_root = download()
print(f'Dataset kökü: {dataset_root}')

## Hücre 5 — Veri Keşfi / EDA (Faz 1)

In [ ]:
from src.data.explore import main as explore_main
explore_main()

# Grafikleri notebook içinde göster
from IPython.display import Image as IPImage, display
for fig in ['class_distribution.png', 'sample_grid.png', 'images_per_patient.png']:
    display(IPImage(f'experiments/figures/{fig}'))

## Hücre 6 — Hasta-Bazlı Train/Val/Test Bölme (Faz 1)

In [ ]:
from src.data.split import main as split_main
split_main()

# Bölme özetini göster
import pandas as pd
print(pd.read_csv('experiments/tables/split_summary.csv').to_string(index=False))

## Hücre 7 — Model Eğitimi (Faz 3 & 4)

In [ ]:
# Faz 2 eğitim hücresi — tüm 5 modeli sırayla eğit + test et.
# Tek bir model için: --model resnet50  (veya baseline, classical_ml, alexnet, vgg16)
# Sadece pipeline testı (CPU, 1 epoch): --smoke-test ekleyin

import subprocess, sys

# --- Tüm modelleri eğit (önerilen) ---
subprocess.run(
    [sys.executable, "-m", "src.train",
     "--model", "all",
     "--config", "config.yaml"],
    check=True,
)

# --- Tek model eğitmek için yorum satırını kaldırın ---
# MODEL = "resnet50"  # baseline | classical_ml | alexnet | vgg16 | resnet50
# subprocess.run(
#     [sys.executable, "-m", "src.train",
#      "--model", MODEL,
#      "--config", "config.yaml"],
#     check=True,
# )


## Hücre 8 — Değerlendirme & Karşılaştırma (Faz 5)

In [ ]:
# Faz 5 tamamlandıktan sonra aktif edin.

# import subprocess
# subprocess.run(['python', 'scripts/run_test.py', '--all'], check=True)
# subprocess.run(['python', '-m', 'src.analysis.compare'], check=True)
# subprocess.run(['python', '-m', 'src.analysis.ablation'], check=True)

# # Karşılaştırma tablosunu göster
# import pandas as pd
# print(pd.read_csv('experiments/tables/comparison_table.csv').to_string(index=False))

print('Faz 5 tamamlandıktan sonra bu hücreyi aktif edin.')